# Proyecto BATSÚ — Notebook 02: Generación del corpus estructurado

**Universidad Cenfotec — MISIA** · Alvaro Granados, Jose Leonel Hernández

Este notebook ejecuta la fase de **simulación controlada** (Sección 3.4): corre cada una de las 60 semillas de la Tabla 4.6 un número fijo de veces contra el simulador construido en el notebook 01, con la política heurística de exploración ε = 0.20 (decisión D5), y guarda todas las transiciones (s, a, r, s', terminated, truncated) en un archivo Parquet en `corpus/`. Ese archivo es el *Experience Replay Buffer* inicial (instrumento 3.6.4) y el insumo de los notebooks 03 (texto), 04 (juez) y 05 (entrenamiento).

No interviene ningún modelo de lenguaje: todo lo decide la tabla de reacción (decisión D3). Por eso la corrida completa tarda menos de un minuto y se puede repetir cuantas veces haga falta.

**Secciones**
1. Configuración e importación del simulador
2. Parámetros de la corrida
3. Generación de las transiciones
4. Verificación: distribuciones, longitud y fidelidad al prior (decisión D1)
5. Guardado del corpus y del manifiesto


## 1. Configuración e importación del simulador

Se monta Drive, se localiza la carpeta del proyecto y se importa `batsu_sim.py`, el módulo que el notebook 01 escribió con el ambiente, la tabla de reacción y las políticas. Si el módulo no existe, hay que correr el notebook 01 completo primero.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, json, time, hashlib
import numpy as np
import pandas as pd

NOMBRE_CARPETA = 'DatosBatsuv2'
candidatos = [p for p in Path('/content/drive/MyDrive').rglob(NOMBRE_CARPETA) if p.is_dir()]
assert candidatos, f'No se encontró la carpeta "{NOMBRE_CARPETA}"'
BASE_DIR = candidatos[0]
CORPUS_DIR = BASE_DIR / 'corpus'
CORPUS_DIR.mkdir(exist_ok=True)

!pip install -q gymnasium pyarrow 2>&1 | tail -1
sys.path.insert(0, str(BASE_DIR))
assert (BASE_DIR / 'batsu_sim.py').exists(), 'Falta batsu_sim.py: correr el notebook 01 completo (última celda) primero'
import batsu_sim as bs
print('Simulador importado desde', BASE_DIR, '· semillas:', len(bs.semillas), '· acciones:', len(bs.ACCIONES))


Mounted at /content/drive
Parámetros cargados. Acciones: 7 | Estados afectivos: 5
Semillas: 60 OK · distancia máxima 267.1 km · inviables: ['CR-TUR-005', 'CR-TUR-020', 'CR-TUR-035', 'CR-TUR-044', 'CR-TUR-054']
Tabla de reacción: 35 celdas OK
Modificadores: 7 OK
Dinámica OK. Desenlaces al cerrar (entusiasmado, oferta baja, prior=conversión):
{'conversion': 0.82, 'avance_parcial': 0.144, 'abandono': 0.036} | esperado: conv (75+40)/140=0.82, avance 0.14, abandono 0.04
Ambiente creado · Box(0.0, 1.0, (21,), float32) · Discrete(7)
Máscara Trust Gate para CR-TUR-001 (presupuesto bajo, solo texto): {'solicitar_informacion': True, 'ofrecer_paquete_bajo': True, 'ofrecer_paquete_medio': False, 'ofrecer_paquete_alto': False, 'enviar_activo': False, 'escalar_humano': True, 'cerrar_conversacion': True}
Simulador importado desde /content/drive/MyDrive/Proyecto de investigacion/DatosBatsuv2 · semillas: 60 · acciones: 7


## 2. Parámetros de la corrida

- **`CORRIDAS_POR_SEMILLA`**: cuántas veces se ejecuta cada semilla. El documento (Sección 3.5) fija 50; con la escala de cierre 0.5 y ~5 turnos por episodio, 100 corridas producen las ~30 000 transiciones previstas. Para una prueba rápida basta con 5.
- **`PRIOR_PUNTOS`**: puntos que el modificador M8 suma al desenlace esperado (decisión D1). El documento fija 20; se deja como parámetro porque la verificación de la Sección 4 mostró que con 20 la fidelidad al prior queda por debajo del rango aprobado, y 40 la sitúa dentro.
- **`ESCALA_P_CIERRE`**: factor sobre la probabilidad de cierre de la Tabla 5.11.a (1.0 = tabla original). Se sobreescribe en el módulo para que este notebook controle el valor sin depender de lo que tenga guardado `batsu_sim.py`.
- **`ETIQUETA`**: nombre de la versión del corpus. Mientras la escala no esté aprobada por el tutor, usar una etiqueta provisional (`v0_escala05`); el corpus definitivo será `v1`.
- **`SEMILLA_GLOBAL`**: fija el generador aleatorio. Cada corrida recibe una semilla derivada de (semilla global, índice de la semilla, número de corrida), de modo que cualquier episodio se puede reproducir individualmente.


In [2]:
CORRIDAS_POR_SEMILLA = 100
ESCALA_P_CIERRE = 0.5
PRIOR_PUNTOS = 40                                  # D1: 20 inicial (fidelidad 43 %); 40 aprobado (Sección 5.4.6)
ETIQUETA = 'v0_escala05_prior40'
SEMILLA_GLOBAL = bs.SEMILLA_ALEATORIA_GLOBAL      # 20260925, definido en el notebook 01
EPSILON = bs.EPSILON_HEURISTICA                   # 0.20, decisión D5

assert hasattr(bs, 'ESCALA_P_CIERRE'), 'batsu_sim.py no tiene ESCALA_P_CIERRE: agregarla en el notebook 01 y regenerar el módulo'
bs.ESCALA_P_CIERRE = ESCALA_P_CIERRE
bs.PRIOR_PUNTOS = PRIOR_PUNTOS
print(f'Corridas por semilla: {CORRIDAS_POR_SEMILLA} · escala p(cierre): {bs.ESCALA_P_CIERRE} · prior: {bs.PRIOR_PUNTOS} pts · ε: {EPSILON} · etiqueta: {ETIQUETA}')
print(f'Episodios previstos: {60 * CORRIDAS_POR_SEMILLA}')


Corridas por semilla: 100 · escala p(cierre): 0.5 · prior: 40 pts · ε: 0.2 · etiqueta: v0_escala05_prior40
Episodios previstos: 6000


## 3. Generación de las transiciones

Para cada semilla y cada corrida se crea un generador aleatorio propio, se corre un episodio completo con la política heurística (ε = 0.20) y se registra cada turno como una fila. Cada fila contiene:

- identificación: `id_semilla`, `corrida`, `turno`, `semilla_rng`;
- la transición del MDP: las 21 dimensiones de s (columnas s_*, incluido s_turno_norm), la acción, la recompensa total y sus componentes (r_shaping, r_trust, r_juez, r_terminal), las 21 dimensiones de s' (columnas sp_*), terminated, truncated;
- contexto para el registro y para la redacción posterior del texto: estado afectivo y etapa AIDA antes y después, modificadores activados, activo enviado, violación del Trust Gate, desenlace.

La columna **`r_juez`** queda en 0 (decisión D6) y se completará en el notebook 04 mediante un `UPDATE` por `id_transicion`, sin regenerar el corpus. Por eso cada fila lleva un identificador único.


In [3]:
env = bs.BatsuEnv(bs.semillas)
filas = []
t0 = time.time()

for i_sem, id_sem in enumerate(env.ids):
    for corrida in range(CORRIDAS_POR_SEMILLA):
        seed = int(np.random.SeedSequence([SEMILLA_GLOBAL, i_sem, corrida]).generate_state(1)[0])
        obs, _ = env.reset(seed=seed, options={'id': id_sem})
        rng_pol = np.random.default_rng(seed + 1)          # generador independiente para la exploración
        while True:
            a = bs.politica_heuristica(env, EPSILON, rng_pol)
            obs2, r, term, trunc, info = env.step(a)
            fila = {'id_transicion': f'{id_sem}-{corrida:03d}-{info["turno"]:02d}',
                    'id_semilla': id_sem, 'corrida': corrida, 'semilla_rng': seed, 'turno': info['turno'],
                    **{f's_{n}': float(v) for n, v in zip(bs.NOMBRES_DIM, obs)},
                    'accion': a, 'accion_nombre': info['accion_nombre'], 'reward': r,
                    'r_shaping': info['r_shaping'], 'r_trust': info['r_trust'], 'r_juez': info['r_juez'], 'r_terminal': info['r_terminal'],
                    **{f'sp_{n}': float(v) for n, v in zip(bs.NOMBRES_DIM, obs2)},
                    'terminated': term, 'truncated': trunc,
                    'estado_antes': info['estado_antes'], 'estado_despues': info['estado_despues'],
                    'etapa_antes': info['etapa_antes'], 'etapa_despues': info['etapa_despues'],
                    'p_cierre': info['p_cierre'], 'modificadores': info['modificadores'], 'activo_enviado': info['activo_enviado'],
                    'violacion_trust_gate': info['violacion_trust_gate'], 'enmascarada': info['enmascarada'],
                    'desenlace': info['desenlace']}
            filas.append(fila)
            obs = obs2
            if term or trunc:
                break

corpus = pd.DataFrame(filas)
n_ep = corpus.groupby(['id_semilla', 'corrida']).ngroups
print(f'{len(corpus):,} transiciones en {n_ep:,} episodios · {len(corpus)/n_ep:.2f} turnos por episodio · {time.time()-t0:.1f} s')


26,343 transiciones en 6,000 episodios · 4.39 turnos por episodio · 6.9 s


## 4. Verificación

Antes de guardar se verifica que el corpus tenga la estructura esperada y se calculan los indicadores que el documento pide reportar:

- **Cobertura**: 60 semillas × N corridas, sin episodios faltantes; cada episodio termina exactamente una vez.
- **Longitud**: turnos promedio y distribución (para contrastar con la Sección 3.5).
- **Desenlaces**: distribución global y por banda de distancia.
- **Fidelidad al prior (decisión D1)**: fracción de semillas cuyo desenlace modal (el más frecuente entre sus corridas) coincide con el resultado esperado. El criterio aprobado por el tutor exige que esté entre 50 % y 65 %: por encima, el prior domina y elimina la varianza que el agente necesita; por debajo, el prior se pierde.
- **Trust Gate**: proporción de turnos con violación y de acciones enmascaradas, que son los ejemplos negativos que introduce la exploración.
- **Acciones**: distribución de las acciones tomadas, para confirmar que la exploración cubre las siete.


In [4]:
ep = corpus.groupby(['id_semilla', 'corrida']).agg(turnos=('turno', 'max'), retorno=('reward', 'sum'),
                                                   desenlace=('desenlace', 'last'), fin=('terminated', 'sum'), tr=('truncated', 'sum')).reset_index()
assert len(ep) == 60 * CORRIDAS_POR_SEMILLA, 'Faltan episodios'
assert ((ep['fin'] + ep['tr']) == 1).all(), 'Hay episodios que no terminan exactamente una vez'
assert corpus['id_transicion'].is_unique
assert corpus['desenlace'].notna().sum() == len(ep), 'El desenlace debe aparecer solo en la última transición'

print(f'Turnos por episodio: media {ep.turnos.mean():.2f} · mediana {ep.turnos.median():.0f} · máx {ep.turnos.max()}')
print('Distribución de turnos:', ep.turnos.value_counts().sort_index().to_dict())
print('Retorno promedio por episodio:', round(ep.retorno.mean(), 2))
print()
print('Desenlaces (global):', ep.desenlace.value_counts(normalize=True).round(3).to_dict())

# Fidelidad al prior (D1): desenlace modal por semilla vs resultado esperado
esperado = bs.semillas.set_index('id')['resultado_esperado']
modal = ep.groupby('id_semilla')['desenlace'].agg(lambda s: s.value_counts().idxmax())
fid = pd.DataFrame({'esperado': esperado, 'modal': modal})
fid['coincide'] = fid['esperado'] == fid['modal']
fidelidad = fid['coincide'].mean()
ok = bs.PRIOR_FIDELIDAD_MIN <= fidelidad <= bs.PRIOR_FIDELIDAD_MAX
print()
print(f'Fidelidad al prior (D1): {fidelidad:.1%} de las semillas · rango aprobado 50–65 % · {"OK" if ok else "FUERA DE RANGO"}')
print('Coincidencia por resultado esperado:', fid.groupby('esperado')['coincide'].mean().round(2).to_dict())
print()
print(f'Turnos con violación del Trust Gate: {corpus.violacion_trust_gate.mean():.1%} · enmascarados: {corpus.enmascarada.mean():.1%}')
print('Acciones tomadas:', corpus['accion_nombre'].value_counts(normalize=True).round(3).to_dict())
print()
banda = bs.semillas.set_index('id')['distancia_categoria_calc']
print('Desenlaces por banda de distancia:')
print(pd.crosstab(ep['id_semilla'].map(banda), ep['desenlace'], normalize='index').round(2))


Turnos por episodio: media 4.39 · mediana 4 · máx 12
Distribución de turnos: {1: 804, 2: 877, 3: 1146, 4: 882, 5: 623, 6: 449, 7: 360, 8: 223, 9: 187, 10: 132, 11: 101, 12: 216}
Retorno promedio por episodio: 4.4

Desenlaces (global): {'conversion': 0.478, 'avance_parcial': 0.248, 'abandono': 0.115, 'rechazo': 0.068, 'recomendacion_inviable': 0.064, 'finalizacion_limite_turnos': 0.028}

Fidelidad al prior (D1): 60.0% de las semillas · rango aprobado 50–65 % · OK
Coincidencia por resultado esperado: {'abandono': 0.4, 'avance_parcial': 0.4, 'conversion': 1.0, 'finalizacion_limite_turnos': 0.0, 'rechazo': 0.2, 'recomendacion_inviable': 1.0}

Turnos con violación del Trust Gate: 5.0% · enmascarados: 0.6%
Acciones tomadas: {'ofrecer_paquete_medio': 0.215, 'enviar_activo': 0.211, 'ofrecer_paquete_bajo': 0.197, 'ofrecer_paquete_alto': 0.185, 'solicitar_informacion': 0.135, 'escalar_humano': 0.03, 'cerrar_conversacion': 0.028}

Desenlaces por banda de distancia:
desenlace   abandono  avance_pa

In [5]:
# ---- Comparación con el corpus anterior (piloto con vector de 20 componentes) ----
# Si los episodios son idénticos, los textos (03b) y juicios (04) ya generados siguen siendo válidos,
# porque se enlazan por id_transicion. Las únicas columnas nuevas deben ser s_turno_norm y sp_turno_norm.
ruta_anterior = CORPUS_DIR / f'corpus_{ETIQUETA}.parquet'
if ruta_anterior.exists():
    anterior = pd.read_parquet(ruta_anterior)
    nuevas = sorted(set(corpus.columns) - set(anterior.columns))
    quitadas = sorted(set(anterior.columns) - set(corpus.columns))
    print('Columnas nuevas:', nuevas, '· columnas quitadas:', quitadas)
    comunes = [c for c in anterior.columns if c in corpus.columns and c != 'r_juez']
    mismo_largo = len(anterior) == len(corpus)
    identicos = mismo_largo and anterior[comunes].reset_index(drop=True).equals(corpus[comunes].reset_index(drop=True))
    print(f'Transiciones: anterior {len(anterior):,} · nuevo {len(corpus):,}')
    print('Episodios idénticos al corpus anterior:', 'SÍ' if identicos else 'NO')
    if not identicos and mismo_largo:
        dif = [c for c in comunes if not anterior[c].reset_index(drop=True).equals(corpus[c].reset_index(drop=True))]
        print('Columnas con diferencias:', dif)
else:
    print('No hay corpus anterior con esta etiqueta; se genera por primera vez.')

Columnas nuevas: ['s_turno_norm', 'sp_turno_norm'] · columnas quitadas: []
Transiciones: anterior 26,343 · nuevo 26,343
Episodios idénticos al corpus anterior: SÍ


## 5. Guardado del corpus y del manifiesto

Se guarda el corpus en Parquet (compacto, tipado, legible por pandas y por Stable-Baselines3) y, junto a él, un **manifiesto** JSON con todo lo necesario para reproducir la corrida: parámetros, versiones de librerías, hash de los tres CSV de entrada, conteos y los indicadores de verificación. El manifiesto es el registro que el documento exige para el corpus (Sección 5.5) y la evidencia que se reporta en el capítulo de resultados.

También se guarda la tabla de fidelidad por semilla, útil para la revisión con el tutor.


In [6]:
import shutil

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

ruta_corpus = CORPUS_DIR / f'corpus_{ETIQUETA}.parquet'
ruta_manif = CORPUS_DIR / f'manifiesto_{ETIQUETA}.json'
ruta_fid = CORPUS_DIR / f'fidelidad_prior_{ETIQUETA}.csv'
ruta_sinjuez = CORPUS_DIR / f'corpus_{ETIQUETA}_sinjuez.parquet'

# Respaldo del corpus anterior (una sola vez): conserva la evidencia del piloto de 20 componentes
manif_anterior = json.load(open(ruta_manif, encoding='utf-8')) if ruta_manif.exists() else {}
dim_anterior = sum(1 for c in pd.read_parquet(ruta_corpus).columns if c.startswith('s_')) if ruta_corpus.exists() else None
if dim_anterior is not None and dim_anterior != len(bs.NOMBRES_DIM):
    respaldo = CORPUS_DIR / f'piloto_{dim_anterior}dim'
    respaldo.mkdir(exist_ok=True)
    for p in [ruta_corpus, ruta_manif, ruta_fid, ruta_sinjuez]:
        if p.exists() and not (respaldo / p.name).exists():
            shutil.copy2(p, respaldo / p.name)
    if ruta_sinjuez.exists():
        ruta_sinjuez.unlink()   # la copia sin juez del piloto ya quedó respaldada; el notebook 04 creará la nueva
    print(f'Piloto de {dim_anterior} componentes respaldado en {respaldo.name}/')

corpus.to_parquet(ruta_corpus, index=False)
fid.to_csv(ruta_fid)

manifiesto = {
    'etiqueta': ETIQUETA, 'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
    'parametros': {'corridas_por_semilla': CORRIDAS_POR_SEMILLA, 'escala_p_cierre': ESCALA_P_CIERRE, 'epsilon': EPSILON,
                   'semilla_global': SEMILLA_GLOBAL, 'turno_limite': bs.TURNO_LIMITE, 'r_violacion_trust_gate': bs.R_VIOLACION_TRUST_GATE,
                   'prior_puntos': PRIOR_PUNTOS, 'dim_estado': len(bs.NOMBRES_DIM)},
    'entradas': {f: sha256(bs.DATOS_DIR / f) for f in ['semillas_maestra.csv', 'tabla_reaccion.csv', 'modificadores_reaccion.csv']},
    'codigo': {'batsu_sim.py': sha256(BASE_DIR / 'batsu_sim.py')},
    'versiones': {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
                  'gymnasium': __import__('gymnasium').__version__},
    'resultados': {'transiciones': int(len(corpus)), 'episodios': int(len(ep)), 'turnos_promedio': round(float(ep.turnos.mean()), 3),
                   'retorno_promedio': round(float(ep.retorno.mean()), 3),
                   'desenlaces': ep.desenlace.value_counts(normalize=True).round(4).to_dict(),
                   'fidelidad_prior': round(float(fidelidad), 4), 'fidelidad_en_rango': bool(ok),
                   'tasa_violacion_trust_gate': round(float(corpus.violacion_trust_gate.mean()), 4)},
    # El corpus recién generado tiene r_juez = 0; el notebook 04 vuelve a aplicar los juicios y marca completado.
    'juez': {'completado': False, 'modelo': manif_anterior.get('juez', {}).get('modelo'),
             'nota': 'r_juez = 0 tras regenerar el corpus (D6); el notebook 04 aplica los juicios por id_transicion'},
}
# Se conservan los bloques que escriben otros notebooks (03b: generador), en vez de borrarlos
for clave in ['generador']:
    if clave in manif_anterior:
        manifiesto[clave] = manif_anterior[clave]

json.dump(manifiesto, open(ruta_manif, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('Guardado:', ruta_corpus.name, f'({ruta_corpus.stat().st_size/1e6:.1f} MB)', '·', ruta_manif.name, '·', ruta_fid.name)
print('Bloques conservados del manifiesto anterior:', [k for k in ['generador'] if k in manifiesto])

Piloto de 20 componentes respaldado en piloto_20dim/
Guardado: corpus_v0_escala05_prior40.parquet (0.5 MB) · manifiesto_v0_escala05_prior40.json · fidelidad_prior_v0_escala05_prior40.csv
Bloques conservados del manifiesto anterior: ['generador']
